In [1]:
%pip install yfinance statsmodels

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import yfinance as yf

from statsmodels.tsa.stattools import coint

tickers = ["KO", "PEP", "PG", "CL", "KMB", "GIS", "KHC"]

prices = yf.download(
    tickers,
    start="2018-01-01",
    end="2026-10-07",
    auto_adjust=True,
    progress=False
)["Close"]

prices = prices.dropna(how="all")

print(prices.tail())
print("Available tickers:", prices.columns.tolist())

Ticker             CL        GIS        KHC        KMB         KO         PEP  \
Date                                                                            
2026-09-30  85.320000  32.160000  22.719999  97.430000  86.080002  126.720001   
2026-10-01  84.529999  31.670000  22.459999  94.339996  86.099998  125.599998   
2026-10-02  84.260002  32.009998  22.190001  94.360001  85.650002  125.889999   
2026-10-05  86.190002  31.680000  21.840000  95.410004  86.510002  125.650002   
2026-10-06  87.250000  32.180000  22.030001  96.769997  86.169998  125.709999   

Ticker              PG  
Date                    
2026-09-30  145.279999  
2026-10-01  143.949997  
2026-10-02  144.910004  
2026-10-05  145.940002  
2026-10-06  148.410004  
Available tickers: ['CL', 'GIS', 'KHC', 'KMB', 'KO', 'PEP', 'PG']


In [3]:
formation = prices.loc[:"2021-12-31"]

screen_results = []

for ticker_a, ticker_b in combinations(
    formation.columns.tolist(), 2
):
    pair = formation[[ticker_a, ticker_b]].dropna()

    if len(pair) < 500:
        continue

    log_pair = np.log(pair)

    stat, pvalue, critical_values = coint(
        log_pair[ticker_a],
        log_pair[ticker_b],
        trend="c",
        autolag="aic"
    )

    beta = np.polyfit(
        log_pair[ticker_b].to_numpy(),
        log_pair[ticker_a].to_numpy(),
        1
    )[0]

    screen_results.append({
        "ticker_a": ticker_a,
        "ticker_b": ticker_b,
        "observations": len(pair),
        "beta": beta,
        "coint_stat": stat,
        "p_value": pvalue,
        "critical_5pct": critical_values[1]
    })

screen_df = pd.DataFrame(screen_results)

screen_df = screen_df.sort_values("p_value")

print(screen_df.to_string(index=False))

ticker_a ticker_b  observations      beta  coint_stat  p_value  critical_5pct
      CL      GIS          1008  0.588396   -3.464092 0.035714      -3.342204
     GIS      KMB          1008  1.108085   -3.397284 0.042653      -3.342204
     KMB       PG          1008  0.529152   -3.387622 0.043744      -3.342204
     GIS      PEP          1008  0.965890   -3.251166 0.061780      -3.342204
     GIS       PG          1008  0.633800   -3.179581 0.073409      -3.342204
      KO      PEP          1008  0.686537   -3.046935 0.099466      -3.342204
      KO       PG          1008  0.433426   -2.970225 0.117454      -3.342204
      CL       PG          1008  0.397330   -2.763207 0.177672      -3.342204
     PEP       PG          1008  0.632468   -2.676454 0.208161      -3.342204
      CL      PEP          1008  0.619290   -2.572824 0.247929      -3.342204
     KMB      PEP          1008  0.762368   -2.550778 0.257145      -3.342204
     KHC      KMB          1008 -1.267019   -2.468337 0.293209  